In [0]:
##  Packages  ###############################

import pip
!pip install OSGridConverter
!pip install folium
!pip install pyproj
!pip install shapely
!pip install os-grid-reference
!pip install geopandas

from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import numpy as np
from pyproj import Proj, transform, Transformer
import json
from shapely.geometry import shape, GeometryCollection, Point
import branca.colormap
import OSGridConverter
import folium
from folium import plugins
from folium.plugins import HeatMap
from os_grid_reference import ngr_decode
import geopandas
import pandas as pd

In [0]:
##  Variables  #####################################################

root = ""
mapStyle = "Country"
metric = "Total_value"
mapDataLoc = root+"world.geo.json"
featureName = "name"
analyticalDataLoc = root+"sample_data"

In [0]:
## Load geographical data  ###########################################

# Helper function to convert eastings/northings to latitude/longitude
def convertEastNorthToLatLong(data):
    transformer = Transformer.from_crs("EPSG:27700", "EPSG:4326")
    for feature in data['features']:

        polygon = feature['geometry']

        if polygon["type"]=="Polygon":
            coords = polygon['coordinates']
            for coordList in coords:
                for coordPair in coordList:
                    x1 = coordPair[0]
                    y1 = coordPair[1]
                    coordPair[1],coordPair[0] = transformer.transform(x1,y1)
        else:
            coords = polygon['coordinates']
            for coordListList in coords:
                for coordList in coordListList:
                    for coordPair in coordList:
                        x1 = coordPair[0]
                        y1 = coordPair[1]
                        coordPair[1],coordPair[0] = transformer.transform(x1,y1)
    
# Load in map data
with open(mapDataLoc, 'r') as f:
    data = json.load(f)


In [0]:
import os

os.chdir("..")
print(os.listdir())

In [0]:
##  Load in analytical data  ###########################################

df = pd.read_csv(analyticalDataLoc)

In [0]:
##  Create map  ############################################################

# Initialise map
m = folium.Map(location=[51.5074, 0.1278],
                    zoom_start = 3)

# Set the index as 'Constituency', 'County' or 'Water company'
df_indexed = df.set_index(mapStyle)

# Set the colormap (use scale going from 0 to 100 for percentage)
colormap_linear = branca.colormap.LinearColormap(['#A285D1', '#12436D'], index = None,\
    vmin=0, vmax=np.log(max(df_indexed.loc[:, metric])),\
    caption=metric, max_labels=100, tick_labels=[0, round(max(df_indexed.loc[:, metric])/2, -6), round(max(df_indexed.loc[:, metric]), -6)])

c_linear = np.linspace(0, np.log(max(df_indexed.loc[:, metric])), 20)
c_log = np.exp(c_linear)

colors = list(map(lambda x: colormap_linear(x)[:7], c_linear))

colormap = branca.colormap.LinearColormap(colors, index = c_log,\
    vmin=0, vmax=max(df_indexed.loc[:, metric]),\
    caption=metric, max_labels=100, tick_labels=[0, round(max(df_indexed.loc[:, metric])/2, -6), round(max(df_indexed.loc[:, metric]), -6)])

# For whatever is being plotted, add the colour values to the Folium map
cp = folium.GeoJson(
    data,
    name=metric,
    style_function=lambda feature: {
        "fillColor": colormap(df_indexed.loc[feature['properties'][featureName], metric])
        if feature["properties"][featureName] in df_indexed.index
        else "white",
        "color": "black",
        "fillOpacity": 0.9,
    },
).add_to(m)

# Now add the data to the Folium map
for s in cp.data['features']:
    if s['properties'][featureName] in df_indexed.index:
        s['properties'][metric] = int(df_indexed.loc[s['properties'][featureName], metric])
    else:
        s['properties'][metric] = 0

# Caption the colorbar as the metric of interest then add to the map
colormap.caption = metric
colormap.add_to(m)

# Add a tooltip showing the region name and its metric value, displayed when the mouse hovers over the area
tooltip = folium.GeoJsonTooltip(
    fields=[featureName, metric],
    aliases=[mapStyle+":", metric+":"],
    localize=True,
    sticky=False,
    labels=True,
    style="""
        background-color: #F0EFEF;
        border: 2px solid black;
        border-radius: 3px;
        box-shadow: 3px;
          """,
    max_width=800,
)
tooltip.add_to(cp)

# Only relevant if plotting multiple things on top of each other
# folium.LayerControl().add_to(m)

# Save as html
m.save(root+mapStyle+'.html')

display(m)

In [0]:
colormap(1000000000)
print(c_index)
print(colors)